<a href="https://colab.research.google.com/github/arsitekberotot/arsitrad/blob/main/legacy/Arsitrad_Gemma4_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arsitrad — Fine-Tune Gemma 4 2B for Indonesian Building Regulations
## RAG-Powered AI Chatbot for Indonesian Construction Law

**Author:** Hanif | UNDIP Architecture  
**Repo:** https://github.com/arsitekberotot/arsitrad

---

## Overview

This notebook fine-tunes **Gemma 4 2B E2B** with QLoRA on Indonesian building regulations using:
- **151 instruction-tuning examples** from `data/processed/arsitrad_training.jsonl`
- **ChromaDB RAG** for grounded generation
- **Unsloth** for 2x faster fine-tuning

**Runtime Requirements:**
| Section | GPU Required | Time |
|---------|-------------|------|
| Sections 1-3 (Setup + Data) | T4 (free) | ~10 min |
| Section 4 (Fine-tuning) | **A100** (Colab Pro) | ~30-60 min |

---

## Table of Contents
1. [Setup & Install](#1)
2. [Clone Repo & Load Training Data](#2)
3. [Test RAG Retrieval (No GPU)](#3)
4. [Fine-Tune Gemma 4 with QLoRA](#4)
5. [Test Fine-Tuned Model](#5)
6. [Push to HuggingFace (Optional)](#6)



In [1]:
# Install git-lfs for large file support
!apt-get install -y git-lfs > /dev/null 2>&1
!git lfs install
print('git-lfs installed')

Git LFS initialized.
git-lfs installed


<a name='1'></a>
## 1. Setup & Installation


In [2]:
# Install dependencies
!pip install -q --upgrade pip
!pip install -q \
    torch \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    bitsandbytes \
    accelerate>=0.27.0 \
    scipy \
    chromadb>=0.4.0 \
    sentence-transformers>=2.5.0 \
    datasets>=2.16.0 \
    trl>=0.7.0 \
    gradio>=4.0.0 \
    pyyaml \
    tqdm \
    huggingface_hub

# Install Unsloth for 2x faster fine-tuning
!pip install -q unsloth

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Fine-tuning requires GPU.")


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4 (15.6 GB)


<a name='2'></a>
## 2. Clone Repo & Load Training Data


In [3]:
# Clone your GitHub repo
!git clone https://github.com/arsitekberotot/arsitrad.git
%cd arsitrad

# Verify files
import os
print("Top-level contents:")
for f in os.listdir('.'):
    print(f"  {f}")
print()
print("Training data:")
for f in sorted(os.listdir('data/processed')):
    sz = os.path.getsize(f'data/processed/{f}')
    print(f"  {f}: {sz/1024:.1f} KB")


Cloning into 'arsitrad'...
remote: Enumerating objects: 289, done.
remote: Counting objects: 100% (289/289), done.
remote: Compressing objects: 100% (253/253), done.
remote: Total 289 (delta 39), reused 281 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (289/289), 14.93 MiB | 13.61 MiB/s, done.
Resolving deltas: 100% (39/39), done.
Filtering content: 100% (16/16), 497.51 MiB | 15.36 MiB/s, done.
/content/arsitrad
Top-level contents:
  docker
  data
  requirements.txt
  .gitignore
  .git
  Arsitrad_Gemma4_Indonesian_Architecture_AI.ipynb
  README.md
  rag
  fine-tune
  .gitattributes
  ui
  scraper
  agent
  reasoning_db
  Arsitrad_Gemma4_FineTune.ipynb
  verification_report_nodes.md
  demo.py
  config.yaml

Training data:
  arsitrad_training.jsonl: 116.4 KB
  chroma_db: 4.0 KB
  local_chunks.json: 30225.7 KB
  local_embeddings.npy: 55281.1 KB
  national_chunks.json: 17791.1 KB
  national_embeddings.npy: 33973.6 KB


In [4]:
# Load and inspect training dataset
import json
from collections import Counter

with open('data/processed/arsitrad_training.jsonl') as f:
    lines = f.readlines()

print(f"Total training examples: {len(lines)}")

cats = Counter()
for line in lines:
    ex = json.loads(line)
    cats[ex.get('category', 'unknown')] += 1

print("\nCategory distribution:")
for cat, count in cats.most_common():
    print(f"  {cat}: {count}")

ex = json.loads(lines[0])
print(f"\nExample (first 500 chars of 'text' field):")
print(ex['text'][:500])


Total training examples: 151

Category distribution:
  regulation_qa: 139
  function_calling: 4
  disaster_conversation: 2
  settlement_conversation: 2
  local_regulation_conversation: 1
  accessibility_conversation: 1
  permit_conversation: 1
  cooling_conversation: 1

Example (first 500 chars of 'text' field):
<start_of_turn>system
Kamu adalah Arsitrad, asisten AI untuk regulasi dan saran arsitektur di Indonesia. Kamu menjawab berdasarkan peraturan pembangunan Indonesia (UU, PP, Permen, SNI, Perda) dan memberikan saran teknis yang akurat tentang bangunan gedung, tata ruang, dan konstruksi.<end_of_turn>
<start_of_turn>user
 Sebutkan persyaratan untuk kasus ini. (Sumber: UU_6_2023_CiptaKerja)<end_of_turn>
<start_of_turn>model
di dalam negeri yang tidak memenuhi SNI yang
telah diberlakukan secara wajib a


<a name='3'></a>
## 3. Test RAG Retrieval (No GPU Required)


In [5]:
# Setup ChromaDB + embedder
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np

CHROMA_DIR = 'data/processed/chroma_db'
client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.get_collection('arsitrad_national_regulations')
print(f"Collection: {col.name} | Chunks: {col.count()}")

print("Loading embedder...")
model_emb = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print("Ready.")


Collection: arsitrad_national_regulations | Chunks: 22649
Loading embedder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ready.


In [6]:
# Retrieval function with query normalization fix
def retrieve(query, top_k=5, min_similarity=0.3):
    raw_emb = model_emb.encode([query])[0]
    q_norm = raw_emb / np.linalg.norm(raw_emb)
    results = col.query(
        query_embeddings=[q_norm.tolist()],
        n_results=top_k,
        include=['documents', 'metadatas', 'embeddings']
    )
    outputs = []
    for doc, metadata, emb in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['embeddings'][0]
    ):
        if emb is None:
            continue
        stored = np.array(emb)
        cos_sim = float(np.dot(q_norm, stored) / (np.linalg.norm(stored) + 1e-8))
        if cos_sim >= min_similarity:
            outputs.append({'text': doc, 'source': metadata.get('source','?'), 'similarity': round(cos_sim, 4)})
    outputs.sort(key=lambda x: x['similarity'], reverse=True)
    return outputs

# Test
test_queries = [
    'Apa itu KDB dan KDH dalam bangunan gedung?',
    'Syarat izin mendirikan bangunan rumah tinggal',
    'Persyaratan struktur tahan gempa untuk gedung',
]

print("RAG Retrieval Test\n" + "="*50)
for q in test_queries:
    results = retrieve(q, top_k=3)
    print(f"\nQuery: {q}")
    print(f"  Retrieved: {len(results)} chunks")
    for r in results[:2]:
        print(f"    [{r['source']}] sim={r['similarity']}")
        print(f"    {r['text'][:100]}...")


RAG Retrieval Test

Query: Apa itu KDB dan KDH dalam bangunan gedung?
  Retrieved: 3 chunks
    [Permen_6_2025_StandarKegiatan] sim=0.4151
    
PELAKSANAAN 
PENGAWASAN, DAN PENGENAAN SANKSI PADA 
PENYELENGGARAAN PERIZINAN BERUSAHA 
BERBASIS RI...
    [SNI_2847_2019_PersyaratanBetonStruktural] sim=0.3784
    inkan tegangan geser yang 
setara; dan  
b) Potensi redistribusi kelebihan beban 
lokal 
ke 
pelat 
...

Query: Syarat izin mendirikan bangunan rumah tinggal
  Retrieved: 2 chunks
    [UU_6_2023_CiptaKerja] sim=0.5764
    ngan "perjanjian pendahuluan jual
beli" adalah kesepakatan melakukan jual beli Rumah
yang masih dala...
    [Permen_6_2025_StandarKegiatan] sim=0.4465
    
PELAKSANAAN 
PENGAWASAN, DAN PENGENAAN SANKSI PADA 
PENYELENGGARAAN PERIZINAN BERUSAHA 
BERBASIS RI...

Query: Persyaratan struktur tahan gempa untuk gedung
  Retrieved: 3 chunks
    [SNI_9274_2025_EvaluasiRehabilitasiSeismik] sim=0.7156
    gan dan 
pembungkus dapat efektif digunakan untuk 
memperkuat atau meni

<a name='4'></a>
## 4. Fine-Tune Gemma 4 2B with QLoRA

**GPU Required.** Free T4: reduce epochs to 1, batch_size to 2. A100 (Colab Pro): use 3 epochs, batch_size 4.


In [7]:
# 4.1 Configuration
FINETUNE_CONFIG = {
    'model_name': 'google/gemma-4-E2B-it',
    'max_seq_length': 2048,
    'load_in_4bit': True,
    'lora_rank': 32,
    'lora_alpha': 64,
    'lora_dropout': 0.05,
    'batch_size': 4,
    'gradient_accumulation_steps': 4,
    'num_epochs': 3,
    'learning_rate': 2e-4,
    'warmup_steps': 20,
    'logging_steps': 10,
    'save_steps': 100,
    'output_dir': './arsitrad_finetuned',
    'use_gradient_checkpointing': True,
}

print("Fine-tune Configuration:")
for k, v in FINETUNE_CONFIG.items():
    print(f"  {k}: {v}")

effective = FINETUNE_CONFIG['batch_size'] * FINETUNE_CONFIG['gradient_accumulation_steps']
print(f"\nEffective batch size: {effective}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


Fine-tune Configuration:
  model_name: google/gemma-4-E2B-it
  max_seq_length: 2048
  load_in_4bit: True
  lora_rank: 32
  lora_alpha: 64
  lora_dropout: 0.05
  batch_size: 4
  gradient_accumulation_steps: 4
  num_epochs: 3
  learning_rate: 0.0002
  warmup_steps: 20
  logging_steps: 10
  save_steps: 100
  output_dir: ./arsitrad_finetuned
  use_gradient_checkpointing: True

Effective batch size: 16
GPU: Tesla T4


In [8]:
import unsloth
# 4.2 Load Gemma 4 + Apply LoRA
from unsloth import FastLanguageModel

print("Loading Gemma 4...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FINETUNE_CONFIG['model_name'],
    max_seq_length=FINETUNE_CONFIG['max_seq_length'],
    load_in_4bit=FINETUNE_CONFIG['load_in_4bit'],
    dtype=None,
)
print("Gemma 4 loaded.")

print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=FINETUNE_CONFIG['lora_rank'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                     'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=FINETUNE_CONFIG['lora_alpha'],
    lora_dropout=FINETUNE_CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing=FINETUNE_CONFIG['use_gradient_checkpointing'],
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_2944/2085226235.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Gemma 4...
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Gemma 4 loaded.
Applying LoRA adapters...


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Trainable: 62,078,976 / 4,235,640,352 (1.47%)


In [9]:
# 4.3 Load Dataset & Train
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

print("Loading training dataset...")
dataset = load_dataset('json', data_files='data/processed/arsitrad_training.jsonl', split='train')
print(f"Dataset: {len(dataset)} examples")

print("\nStarting QLoRA Fine-Tuning...")
print(f"  Epochs: {FINETUNE_CONFIG['num_epochs']}")
print(f"  Effective batch: {FINETUNE_CONFIG['batch_size'] * FINETUNE_CONFIG['gradient_accumulation_steps']}")
print(f"  LoRA rank: {FINETUNE_CONFIG['lora_rank']}")

training_args = TrainingArguments(
    output_dir=FINETUNE_CONFIG['output_dir'],
    num_train_epochs=FINETUNE_CONFIG['num_epochs'],
    per_device_train_batch_size=FINETUNE_CONFIG['batch_size'],
    gradient_accumulation_steps=FINETUNE_CONFIG['gradient_accumulation_steps'],
    learning_rate=FINETUNE_CONFIG['learning_rate'],
    weight_decay=0.01,
    warmup_steps=FINETUNE_CONFIG['warmup_steps'],
    logging_steps=FINETUNE_CONFIG['logging_steps'],
    save_steps=FINETUNE_CONFIG['save_steps'],
    save_total_limit=2,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    lr_scheduler_type='cosine',
    report_to='none',
    gradient_checkpointing=FINETUNE_CONFIG['use_gradient_checkpointing'],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=FINETUNE_CONFIG['max_seq_length'],
    dataset_text_field='text',
)

trainer.train()
print("Training complete!")


Loading training dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Dataset: 151 examples

Starting QLoRA Fine-Tuning...
  Epochs: 3
  Effective batch: 16
  LoRA rank: 32


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/151 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 151 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 62,078,976 of 5,185,256,992 (1.20% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
10,12.229644
20,4.548486
30,2.163556


Training complete!


In [6]:
# 4.4 Save Fine-Tuned Model
print("Saving LoRA adapters...")
trainer.save_model(FINETUNE_CONFIG['output_dir'])
tokenizer.save_pretrained(FINETUNE_CONFIG['output_dir'])
print(f"Saved to: {FINETUNE_CONFIG['output_dir']}")

print("Saving merged model (for inference)...")
merged_path = FINETUNE_CONFIG['output_dir'] + '_merged'
model.save_pretrained_merged(merged_path, tokenizer)
print(f"Merged model: {merged_path}")

import os
model_size = sum(
    os.path.getsize(os.path.join(merged_path, f))
    for f in os.listdir(merged_path)
    if os.path.isfile(os.path.join(merged_path, f))
)
print(f"Model size: {model_size / 1024**2:.1f} MB")


Saving LoRA adapters...


NameError: name 'trainer' is not defined

In [8]:
import os

# Check both potential paths
paths_to_check = ['./arsitrad/arsitrad_finetuned', './arsitrad_finetuned']

for path in paths_to_check:
    exists = os.path.exists(path)
    print(f"Checking path: {path} | Exists: {exists}")
    if exists:
        print(f"Contents of {path}:")
        for f in os.listdir(path):
            print(f"  - {f}")
        print("-" * 30)

Checking path: ./arsitrad/arsitrad_finetuned | Exists: True
Contents of ./arsitrad/arsitrad_finetuned:
  - chat_template.jinja
  - processor_config.json
  - adapter_config.json
  - README.md
  - checkpoint-30
  - tokenizer_config.json
  - tokenizer.json
  - adapter_model.safetensors
  - training_args.bin
------------------------------
Checking path: ./arsitrad_finetuned | Exists: False


<a name='5'></a>
## 5. Test Fine-Tuned Model


In [3]:
# Quick inference test — Unsloth Native Loading with separate Tokenizer
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
import torch
import gc
import os

# 1. Clear memory fragmentation
gc.collect()
torch.cuda.empty_cache()

# 2. Path configuration
lora_path = "/content/arsitrad/arsitrad_finetuned"
if not os.path.exists(lora_path):
    lora_path = "./arsitrad_finetuned"

print(f"Using LoRA path: {lora_path}")

# 3. Load model and tokenizer separately
# Note: If already loaded, this will be fast or can be skipped if running partials
print("Loading model with adapters...")
model, _ = FastLanguageModel.from_pretrained(
    model_name = lora_path,
    max_seq_length = 2048,
    load_in_4bit = True,
    dtype = None,
    device_map = {'': 0},
)

print("Loading tokenizer separately...")
tokenizer = AutoTokenizer.from_pretrained(lora_path)

FastLanguageModel.for_inference(model)

# 4. Prompt Setup
SYSTEM = (
    "Kamu adalah Arsitrad, asisten AI untuk regulasi arsitektur Indonesia. "
    "Jawab dalam Bahasa Indonesia, cite sumber regulasi jika relevan."
)

test_prompt = (
    "<start_of_turn>system\n" + SYSTEM + "<end_of_turn>\n"
    "<start_of_turn>user\nApa syarat minimum ventilasi alami untuk rumah tinggal di Indonesia?<end_of_turn>\n"
    "<start_of_turn>model\n"
)

# 5. Generation
print("Generating...")
inputs = tokenizer([test_prompt], return_tensors='pt').to('cuda')
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.3,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nInference Test:")
print("="*50)
print(response.split('model')[-1].strip()[:1500])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Using LoRA path: /content/arsitrad/arsitrad_finetuned
Loading model with adapters...
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Loading tokenizer separately...
Generating...

Inference Test:
Apa syarat minimum ventilasi alami untuk rumah


In [9]:
# Download LoRA adapters from Colab
from google.colab import files
import shutil
import os

# Verified path from our check
lora_path = './arsitrad/arsitrad_finetuned'
zip_path = '/content/arsitrad_finetuned_adapters'

if os.path.exists(lora_path):
    print(f"Found LoRA adapters at {lora_path}. Zipping...")
    shutil.make_archive(zip_path, 'zip', lora_path)
    print(f"Zipped: {zip_path}.zip")
    files.download(f"{zip_path}.zip")
    print("Download started. You can use these weights with the base Gemma 4 model.")
else:
    print(f"Error: {lora_path} not found. Please verify the file path.")

Found LoRA adapters at ./arsitrad/arsitrad_finetuned. Zipping...
Zipped: /content/arsitrad_finetuned_adapters.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started. You can use these weights with the base Gemma 4 model.


<a name='6'></a>
## 6. Push to HuggingFace Hub (Optional)


In [ ]:
# Push to HuggingFace
# Get HF token: https://huggingface.co/settings/tokens

from huggingface_hub import HfApi

HF_TOKEN = 'YOUR_HF_TOKEN_HERE'  # @param {type:"string"}
REPO_NAME = 'arsitrad-gemma4-indonesian-architecture'  # @param {type:"string"}

api = HfApi(token=HF_TOKEN)
repo_id = f'arsitekberotot/{REPO_NAME}'

api.create_repo(repo_id=repo_id, repo_type='model', exist_ok=True)
print(f"Repo: https://huggingface.co/{repo_id}")

api.upload_folder(
    folder_path=merged_path,
    repo_id=repo_id,
    repo_type='model',
)
print(f"Uploaded! Model available at: https://huggingface.co/{repo_id}")
